# Entrega 3 – Reducción de características con SVM

En este cuaderno se realiza un experimento de **selección de características**
usando el modelo SVM que fue seleccionado en la Entrega 2 como mejor clasificador.

Objetivos:

1. Tomar el `features.csv` generado en la Entrega 2.
2. Entrenar un SVM base con **todas** las características.
3. Entrenar variantes con selección de características (`SelectKBest`) para
   K en {15, 30, 60} (puedes ajustar si quieres).
4. Comparar métricas (accuracy y F1-macro) usando una separación
   **estratificada por video** (GroupShuffleSplit).
5. Guardar:
   - `svm_reduced.joblib` (modelo con K óptimo, si vale la pena).
   - `selected_features.json` (nombre de las features seleccionadas).
   - `feature_reduction_summary.md` (resumen de resultados en texto).


## imports y paths

In [7]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, f1_score, classification_report

from dotenv import load_dotenv

import sys

# Para poder importar cosas del proyecto si hace falta
sys.path.append(os.path.abspath(os.path.join("..", "..", "..")))

load_dotenv()

# --------- Detectar ROOT_DIR de forma robusta ---------
def find_project_root(start: Path, max_levels: int = 6) -> Path:
    """
    Sube desde 'start' hasta 'max_levels' niveles buscando una carpeta
    que contenga 'Entrega2' y 'Entrega3'. Si no la encuentra, devuelve 'start'.
    """
    current = start
    for _ in range(max_levels):
        entrega2 = current / "Entrega2"
        entrega3 = current / "Entrega3"
        if entrega2.exists() and entrega3.exists():
            return current
        current = current.parent
    return start

try:
    # Caso script .py
    ROOT_DIR = Path(__file__).resolve()
    ROOT_DIR = find_project_root(ROOT_DIR)
except NameError:
    # Caso Jupyter Notebook
    cwd = Path(os.getcwd()).resolve()
    ROOT_DIR = find_project_root(cwd)

ENTREGA2_DIR = ROOT_DIR / "Entrega2"
ENTREGA3_DIR = ROOT_DIR / "Entrega3"

FEATURES_PATH = ENTREGA2_DIR / "experiments" / "results" / "features.csv"

RESULTS_DIR = ENTREGA3_DIR / "experiments" / "results"
MODELS_DIR = ENTREGA3_DIR / "experiments" / "models"
LOGS_DIR = ENTREGA3_DIR / "experiments" / "logs"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print("📁 ROOT_DIR:", ROOT_DIR)
print("📁 FEATURES_PATH:", FEATURES_PATH)
print("📁 RESULTS_DIR:", RESULTS_DIR)
print("📁 MODELS_DIR:", MODELS_DIR)
print("📁 LOGS_DIR:", LOGS_DIR)


📁 ROOT_DIR: A:\Deployments\IA1_VideoActivityRecognition_ICESI_2025_2
📁 FEATURES_PATH: A:\Deployments\IA1_VideoActivityRecognition_ICESI_2025_2\Entrega2\experiments\results\features.csv
📁 RESULTS_DIR: A:\Deployments\IA1_VideoActivityRecognition_ICESI_2025_2\Entrega3\experiments\results
📁 MODELS_DIR: A:\Deployments\IA1_VideoActivityRecognition_ICESI_2025_2\Entrega3\experiments\models
📁 LOGS_DIR: A:\Deployments\IA1_VideoActivityRecognition_ICESI_2025_2\Entrega3\experiments\logs


## Cargar features.csv

In [8]:
if not FEATURES_PATH.exists():
    raise FileNotFoundError(f"No se encontró features.csv en: {FEATURES_PATH}")

df = pd.read_csv(FEATURES_PATH)
print("Shape de features.csv:", df.shape)
df.head()


Shape de features.csv: (175, 144)


,knee_left_mean,knee_left_std,knee_right_mean,knee_right_std,hip_left_mean,hip_left_std,hip_right_mean,hip_right_std,inclination_mean,inclination_std,...,center_displacement_y_std,displacement_ratio_x_y,t_start,t_end,t_duration,video_progress,frame_start,frame_end,video_id,label
0,2.809319,0.238960,1.963360,0.439324,2.854180,0.144649,2.096193,0.411019,1.448148,0.157631,...,0.051305,0.819675,2.533333,5.200000,2.666667,0.500000,76,156,006beb60-8b9e-4b79-867a-ca1c2f386ce2,sentarse
1,2.722190,0.281661,2.464425,0.362735,2.669825,0.259454,2.423046,0.261321,1.627710,0.033428,...,0.010074,3.410053,0.000000,3.500000,3.500000,0.500000,0,105,011d6524-bb00-49fc-80d5-9e97161ac9ff,ponerse_de_pie
2,3.018146,0.168501,2.892542,0.233494,2.873198,0.132569,3.017637,0.107162,0.048602,0.035068,...,0.033009,0.991129,0.400266,5.203459,4.803193,0.444444,10,130,026c2c6b-d509-4f2d-9535-5cc10e2d5042,caminar
3,3.053953,0.079269,3.004244,0.126800,2.849931,0.127685,2.787469,0.091497,0.119382,0.069325,...,0.043525,1.154699,0.000000,4.846527,4.846527,0.439394,0,145,0358999b-7d10-4fa6-8e90-b55207cbe0ff,sentarse
4,2.981027,0.092635,2.751744,0.161392,3.007329,0.085631,3.002173,0.094994,0.052748,0.035720,...,0.023743,0.817310,0.000000,2.807684,2.807684,0.500000,0,70,058a2a00-c790-4f3c-936a-d3c460e092e5,caminar


## Helper para separar X, y, groups

In [9]:
def split_xy_groups(
    df: pd.DataFrame,
    label_col: str = "label",
):
    """
    Separa X (solo features numéricas), y (labels) y groups (video_id).
    Emula la función _split_xy de Entrega2/src/models/train_models.py
    """
    drop_cols = [label_col, "video_id", "frame_start", "frame_end"]
    X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    # Solo columnas numéricas
    X = X.select_dtypes(include=[np.number]).copy()

    if label_col not in df.columns:
        raise ValueError(f"No se encontró la columna de label: {label_col}")

    y = df[label_col].astype(str).values

    if "video_id" in df.columns:
        groups = df["video_id"].values
    else:
        groups = np.arange(len(df))

    feature_names = X.columns.tolist()
    return X, y, groups, feature_names


X, y, groups, feature_names = split_xy_groups(df)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Número de features:", len(feature_names))


X shape: (175, 140)
y shape: (175,)
Número de features: 140


## Split train/test por video (GroupShuffleSplit)

In [10]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

(train_idx, test_idx,) = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test  = X.iloc[test_idx].reset_index(drop=True)
y_train = y[train_idx]
y_test  = y[test_idx]

print("Tamaño train:", X_train.shape, "– test:", X_test.shape)

classes, counts = np.unique(y, return_counts=True)
print("Clases y conteos globales:")
for c, n in zip(classes, counts):
    print(f"{c}: {n}")


Tamaño train: (140, 140) – test: (35, 140)
Clases y conteos globales:
caminar: 70
girar: 29
ponerse_de_pie: 38
sentarse: 38


## Modelo base SVM (todas las features)

In [11]:
base_clf = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "clf",
            SVC(
                kernel="rbf",
                C=5.0,
                gamma="scale",
                probability=True,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

print("Entrenando SVM base (todas las features)...")
base_clf.fit(X_train, y_train)

y_pred_base = base_clf.predict(X_test)

acc_base = accuracy_score(y_test, y_pred_base)
f1_base = f1_score(y_test, y_pred_base, average="macro")

print(f"✅ SVM base – Accuracy: {acc_base:.3f}, F1-macro: {f1_base:.3f}")
print("\nReporte de clasificación (SVM base):")
print(classification_report(y_test, y_pred_base))


Entrenando SVM base (todas las features)...
✅ SVM base – Accuracy: 0.914, F1-macro: 0.887

Reporte de clasificación (SVM base):
                precision    recall  f1-score   support

       caminar       1.00      1.00      1.00        13
         girar       0.67      1.00      0.80         2
ponerse_de_pie       0.89      0.89      0.89         9
      sentarse       0.90      0.82      0.86        11

      accuracy                           0.91        35
     macro avg       0.86      0.93      0.89        35
  weighted avg       0.92      0.91      0.92        35



## SVM con selección de características (varios K)

In [20]:
from collections import OrderedDict

k_values = [15, 30, 60, 70, 80, 90, 100] 

results = OrderedDict()
models_by_k = {}

for k in k_values:
    print(f"\n🔎 Entrenando SVM con SelectKBest, K={k} ...")
    pipeline = Pipeline(
        [
            ("selector", SelectKBest(score_func=f_classif, k=k)),
            ("scaler", StandardScaler()),
            (
                "clf",
                SVC(
                    kernel="rbf",
                    C=5.0,
                    gamma="scale",
                    probability=True,
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
        ]
    )

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")

    print(f"   → Accuracy: {acc:.3f}, F1-macro: {f1:.3f}")

    results[k] = {
        "accuracy": acc,
        "f1_macro": f1,
    }
    models_by_k[k] = pipeline

print("\nResumen comparativo (SVM base vs reducidos):")
print(f"BASE (todas): acc={acc_base:.3f}, f1={f1_base:.3f}")
for k, metrics in results.items():
    print(f"K={k:2d}: acc={metrics['accuracy']:.3f}, f1={metrics['f1_macro']:.3f}")



🔎 Entrenando SVM con SelectKBest, K=15 ...
   → Accuracy: 0.829, F1-macro: 0.775

🔎 Entrenando SVM con SelectKBest, K=30 ...
   → Accuracy: 0.857, F1-macro: 0.839

🔎 Entrenando SVM con SelectKBest, K=60 ...
   → Accuracy: 0.886, F1-macro: 0.861

🔎 Entrenando SVM con SelectKBest, K=70 ...
   → Accuracy: 0.914, F1-macro: 0.887

🔎 Entrenando SVM con SelectKBest, K=80 ...
   → Accuracy: 0.914, F1-macro: 0.887

🔎 Entrenando SVM con SelectKBest, K=90 ...
   → Accuracy: 0.914, F1-macro: 0.887

🔎 Entrenando SVM con SelectKBest, K=100 ...
   → Accuracy: 0.914, F1-macro: 0.887

Resumen comparativo (SVM base vs reducidos):
BASE (todas): acc=0.914, f1=0.887
K=15: acc=0.829, f1=0.775
K=30: acc=0.857, f1=0.839
K=60: acc=0.886, f1=0.861
K=70: acc=0.914, f1=0.887
K=80: acc=0.914, f1=0.887
K=90: acc=0.914, f1=0.887
K=100: acc=0.914, f1=0.887


a:\Deployments\IA1_VideoActivityRecognition_ICESI_2025_2\venv\lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [106 113 114 115 117 118 119 120 121 122] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
a:\Deployments\IA1_VideoActivityRecognition_ICESI_2025_2\venv\lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
a:\Deployments\IA1_VideoActivityRecognition_ICESI_2025_2\venv\lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [106 113 114 115 117 118 119 120 121 122] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
a:\Deployments\IA1_VideoActivityRecognition_ICESI_2025_2\venv\lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


## Elegir mejor K y guardar artefactos

In [21]:
# Elegir el K con mejor F1-macro
best_k = max(results.keys(), key=lambda kk: results[kk]["f1_macro"])
best_model = models_by_k[best_k]

print(f"Mejor modelo reducido: K={best_k} – F1={results[best_k]['f1_macro']:.3f}")

# Extraer nombres de features seleccionadas
selector = best_model.named_steps["selector"]
support_indices = selector.get_support(indices=True)
selected_feature_names = [feature_names[i] for i in support_indices]

print(f"Total de features seleccionadas: {len(selected_feature_names)}")
print("Primeras 10 features seleccionadas:", selected_feature_names[:10])

# Rutas de salida
svm_reduced_path = MODELS_DIR / "svm_reduced.joblib"
selected_features_path = RESULTS_DIR / "selected_features.json"
summary_md_path = RESULTS_DIR / "feature_reduction_summary.md"

import joblib

joblib.dump(best_model, svm_reduced_path)
print(f"Modelo reducido guardado en: {svm_reduced_path}")

with open(selected_features_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "best_k": int(best_k),
            "selected_features": selected_feature_names,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )
print(f"Features seleccionadas guardadas en: {selected_features_path}")

# Guardar resumen en .md para el informe
with open(summary_md_path, "w", encoding="utf-8") as f:
    f.write("# Resumen de reducción de características (SVM)\n\n")
    f.write(f"- F1-macro SVM base (todas las features): {f1_base:.3f}\n")
    f.write(f"- Accuracy SVM base: {acc_base:.3f}\n\n")
    f.write("## Modelos reducidos\n\n")
    for k, metrics in results.items():
        f.write(f"- K={k}: F1-macro={metrics['f1_macro']:.3f}, Accuracy={metrics['accuracy']:.3f}\n")
    f.write("\n")
    f.write(f"Modelo elegido para despliegue reducido: K={best_k}\n")

print(f"Resumen escrito en: {summary_md_path}")


Mejor modelo reducido: K=70 – F1=0.887
Total de features seleccionadas: 70
Primeras 10 features seleccionadas: ['knee_left_mean', 'knee_left_std', 'knee_right_mean', 'knee_right_std', 'hip_left_mean', 'hip_left_std', 'hip_right_mean', 'hip_right_std', 'inclination_std', 'vel_left_hip_mean']
Modelo reducido guardado en: A:\Deployments\IA1_VideoActivityRecognition_ICESI_2025_2\Entrega3\experiments\models\svm_reduced.joblib
Features seleccionadas guardadas en: A:\Deployments\IA1_VideoActivityRecognition_ICESI_2025_2\Entrega3\experiments\results\selected_features.json
Resumen escrito en: A:\Deployments\IA1_VideoActivityRecognition_ICESI_2025_2\Entrega3\experiments\results\feature_reduction_summary.md
